# 03 — Amenity complexity (toy example)

We calculate the **Economic Complexity Index (ECI)** of neighborhoods from the amenities they host,
using the [`ecomplexity`](https://github.com/cid-harvard/py-ecomplexity) package (`pip install ecomplexity`).

The idea: think of neighborhoods as "countries" and amenity types as "products".
- A neighborhood is **diverse** if it hosts many amenity types.
- An amenity is **ubiquitous** if many neighborhoods host it.
- Complex neighborhoods host many amenities, including rare ones that only few other neighborhoods have.

Here we use a small made-up dataset. Later, the same code runs on real data: POIs counted by census tract × Foursquare category.

In [1]:
# ── imports ───────────────────────────────────────────────────────────────────
import pandas as pd
from ecomplexity import ecomplexity

In [2]:
# ── toy data: which amenities can be found in which neighborhood ────────────
# N1 has everything, N6 only has a restaurant
toy = pd.DataFrame({
    "neighborhood": ["N1"] * 6 + ["N2"] * 5 + ["N3"] * 4 + ["N4"] * 3 + ["N5"] * 2 + ["N6"] * 1,
    "amenity": ["restaurant", "grocery", "pharmacy", "gym", "museum", "theater",
                "restaurant", "grocery", "pharmacy", "gym", "museum",
                "restaurant", "grocery", "pharmacy", "gym",
                "restaurant", "grocery", "pharmacy",
                "restaurant", "grocery",
                "restaurant"],
})
toy["present"] = 1  # 1 = the amenity is present in the neighborhood
toy["year"] = 2024  # ecomplexity needs a time column

# ── neighborhood x amenity matrix (just to look at the data) ─────────────────
toy.pivot(index="neighborhood", columns="amenity", values="present").fillna(0).astype(int)

amenity,grocery,gym,museum,pharmacy,restaurant,theater
neighborhood,,,,,,
N1,1,1,1,1,1,1
N2,1,1,1,1,1,0
N3,1,1,0,1,1,0
N4,1,0,0,1,1,0
N5,1,0,0,0,1,0
N6,0,0,0,0,1,0


In [3]:
# ── calculate complexity ──────────────────────────────────────────────────────
# loc = places, prod = amenities, val = presence (0/1)
# presence_test="manual" tells ecomplexity that our values are already 0/1
# with real POI counts we drop this option, and presence is defined by RCA >= 1
cols_input = {"time": "year", "loc": "neighborhood", "prod": "amenity", "val": "present"}
result = ecomplexity(toy, cols_input, presence_test="manual", check_logsupermodularity=False, verbose=False)

result[["neighborhood", "amenity", "mcp", "diversity", "ubiquity", "eci", "pci"]]

,neighborhood,amenity,mcp,diversity,ubiquity,eci,pci
0,N1,grocery,1,6,5,1.46385,-0.29277
1,N1,gym,1,6,3,1.46385,2.04939
2,N1,museum,1,6,2,1.46385,3.22047
3,N1,pharmacy,1,6,4,1.46385,0.87831
4,N1,restaurant,1,6,6,1.46385,-1.46385
5,N1,theater,1,6,1,1.46385,4.39155
6,N2,grocery,1,5,5,0.87831,-0.29277
7,N2,gym,1,5,3,0.87831,2.04939
8,N2,museum,1,5,2,0.87831,3.22047
9,N2,pharmacy,1,5,4,0.87831,0.87831


In [4]:
# ── amenity complexity of neighborhoods (ECI) ────────────────────────────────
result[["neighborhood", "diversity", "eci"]].drop_duplicates().sort_values("eci", ascending=False)

,neighborhood,diversity,eci
0,N1,6,1.46385
6,N2,5,0.87831
12,N3,4,0.29277
18,N4,3,-0.29277
24,N5,2,-0.87831
30,N6,1,-1.46385


In [5]:
# ── complexity of amenities (PCI) ─────────────────────────────────────────────
result[["amenity", "ubiquity", "pci"]].drop_duplicates().sort_values("pci", ascending=False)

,amenity,ubiquity,pci
5,theater,1,4.39155
2,museum,2,3.22047
1,gym,3,2.04939
3,pharmacy,4,0.87831
0,grocery,5,-0.29277
4,restaurant,6,-1.46385
